# Person 3 — Fine-tune WangchanBERTa on Colab

**เป้าหมาย:** ทดลอง pretrained Thai language model แบบ fine-tuning ด้วย train ของทีม และเปรียบเทียบ Macro-F1 / F1 รายคลาสกับ baseline และโมเดลเดิม

1. แตกไฟล์ ZIP โดยคงโครงสร้างโฟลเดอร์ไว้ แล้วเปิด Notebook จากโฟลเดอร์โปรเจกต์ หรือวางโฟลเดอร์ไว้ใน Google Drive (ถ้าใช้ Colab)
2. Notebook จะค้นหา project root จากโครงสร้าง `code/data/` ให้อัตโนมัติ ไม่ต้องแก้ path ในโค้ด
3. ใน Google Colab เลือก Runtime → Change runtime type → GPU (เช่น T4 หากมี) แล้วรันจากบนลงล่าง


ชุด train เท่านั้นที่ใช้ปรับ weights; valid ใช้เลือก checkpoint; sample รายงานหลังเลือกแล้วและมีเพียง 100 แถว; test ใช้สร้าง predictions ด้วยโมเดลที่ทีมฝึก ห้ามติด label เองหรือส่ง test ไปให้ LLM ทำนาย ระวัง valid ถูกใช้เลือกหลายครั้งแล้ว คะแนน valid ไม่ใช่ผลประเมินอิสระ

Notebook นี้ใช้ base checkpoint ไม่ใช้ checkpoint ที่ fine-tune บน sentiment แล้ว เพื่อหลีกเลี่ยงความเสี่ยงข้อมูลชุดประเมินซ้ำกับชุดฝึกของโมเดล sentiment สำเร็จรูป


In [3]:
# Run this cell once in Colab, then continue below.
%pip -q install 'transformers==4.46.3' 'accelerate>=0.26,<2' 'sentencepiece>=0.2,<1' 'pythainlp>=5,<6' 'scikit-learn>=1.4,<2'


รันเซลล์ติดตั้ง packages ด้านบนครั้งแรกก่อนดำเนินการต่อ


In [ ]:
from pathlib import Path

PROJECT_FOLDER_NAME = 'egco461-thai-sentiment'
REQUIRED_DATA_FILES = ('train.csv', 'valid.csv', 'sample_labeled.csv', 'test.csv')

def _is_project_root(path):
    data_path = path / 'code' / 'data'
    return all((data_path / filename).is_file() for filename in REQUIRED_DATA_FILES)

def _find_project_root(search_base):
    search_base = Path(search_base)
    if not search_base.exists():
        return None
    candidates = [search_base, *search_base.parents]
    for depth in range(4):
        candidates.extend(search_base.glob('*/' * depth + PROJECT_FOLDER_NAME))
    checked = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate not in checked and _is_project_root(candidate):
            return candidate
        checked.add(candidate)
    return None

PROJECT_ROOT = _find_project_root(Path.cwd()) or _find_project_root(Path('/content'))

if PROJECT_ROOT is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = _find_project_root(Path('/content/drive/MyDrive'))
    except ImportError:
        pass

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project folder. Extract the project ZIP with its "
        "'code/data/' structure preserved, then run this notebook from that folder "
        "or store the folder in Google Drive."
    )

DATA_DIR = PROJECT_ROOT / 'code' / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'person3_wangchanberta'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
GROUP_ID = 'dingdong'
RUN_ID = 'run3_pos_neu-logit_bias'
SEED = 42
assert all((DATA_DIR / f'{name}.csv').is_file() for name in ['train', 'valid', 'sample_labeled', 'test']), f'ไม่พบ CSV ใน {DATA_DIR}'
print('Data:', DATA_DIR)


In [5]:
import html, re, random, json, platform
import numpy as np
import pandas as pd
import torch
import transformers
from pythainlp.tokenize import word_tokenize
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

set_seed(SEED)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (training will be slow)')
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)
LABELS = ['pos', 'neu', 'neg', 'q']
label2id = {name: i for i, name in enumerate(LABELS)}
id2label = {i: name for name, i in label2id.items()}
frames = {name: pd.read_csv(DATA_DIR / f'{name}.csv') for name in ['train', 'valid', 'sample_labeled', 'test']}
for name, frame in frames.items():
    assert {'id', 'text'}.issubset(frame.columns)
    assert frame['id'].notna().all() and frame['id'].is_unique
    if name != 'test':
        assert frame['category'].notna().all() and set(frame['category']).issubset(LABELS)
    print(name, len(frame))
print(frames['train']['category'].value_counts())


Device: Tesla T4
PyTorch: 2.14.1+cu130 Transformers: 4.46.3
train 24020
valid 2669
sample_labeled 100
test 1577
category
neu    13098
neg     6138
pos     4286
q        498
Name: count, dtype: int64


ใช้ preprocessing แบบย่อ: แปลง HTML entities, รวม whitespace, ตัดคำ `newmm` และแทนช่องว่างด้วย `<_>` ตามรูปแบบที่โมเดลใช้ตอน pretrain โดยยังไม่ลองลบคำซ้ำหรืออีโมจิ บันทึกวิธีนี้ไว้เพื่อทำซ้ำ

In [6]:
def prepare_text(text):
    text = html.unescape('' if pd.isna(text) else str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=True)
    return ''.join('<_>' if token.isspace() else token for token in tokens)

MODEL_NAME = 'airesearch/wangchanberta-base-att-spm-uncased'
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print(prepare_text('ร้านนี้ อร่อยมาก 555'))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

ร้านนี้<_>อร่อยมาก<_>555


In [7]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, frame, include_labels=True):
        self.texts = [prepare_text(text) for text in frame['text']]
        self.encoded = tokenizer(self.texts, truncation=True, max_length=MAX_LENGTH)
        self.labels = [label2id[x] for x in frame['category']] if include_labels else None
    def __len__(self): return len(self.texts)
    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encoded.items()}
        if self.labels is not None: item['labels'] = self.labels[index]
        return item

train_ds = SentimentDataset(frames['train'])
valid_ds = SentimentDataset(frames['valid'])
sample_ds = SentimentDataset(frames['sample_labeled'])
test_ds = SentimentDataset(frames['test'], include_labels=False)
print('Datasets ready')


Datasets ready


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), label2id=label2id, id2label=id2label,
)
# Compute inverse-frequency class weights from train only; valid is never used here.
train_label_ids = np.array([label2id[x] for x in frames['train']['category']])
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.arange(len(LABELS)), y=train_label_ids
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('Class weights:', {label: round(float(class_weights[i]), 4) for i, label in enumerate(LABELS)})
def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple): logits = logits[0]
    pred = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(eval_prediction.label_ids, pred,
        labels=list(range(len(LABELS))), average='macro', zero_division=0)}

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    save_total_limit=2, seed=SEED, data_seed=SEED,
    fp16=torch.cuda.is_available(), report_to='none', logging_steps=100,
)
class WeightedLossTrainer(Trainer):
    def __init__(self, *trainer_args, class_weights=None, **trainer_kwargs):
        super().__init__(*trainer_args, **trainer_kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

trainer = WeightedLossTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
    class_weights=class_weights,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics)
trainer.train()
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best valid Macro-F1:', trainer.state.best_metric)


model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at airesearch/wangchanberta-base-att-spm-uncased and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: {'pos': 1.4011, 'neu': 0.4585, 'neg': 0.9783, 'q': 12.0582}


Epoch,Training Loss,Validation Loss,Macro F1
0,0.764400,0.707873,0.616056
2,0.508400,0.715294,0.647707


Best checkpoint: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/experiments/person3_wangchanberta/checkpoints/checkpoint-3003
Best valid Macro-F1: 0.660798959299878


### ปรับ decision bias สำหรับ pos/neu จาก valid และประเมิน
จาก errors.csv พบว่า neu→pos มากกว่า pos→neu จึงทดลองหักค่าเล็กน้อยจาก logit ของ pos หลังฝึก ไม่ฝึกซ้ำจากตัวอย่างผิด (เพื่อไม่ให้ valid รั่วเข้า train) เลือกค่าจาก valid ด้วย macro-F1 แล้วใช้ค่านั้นกับ sample/test โดยไม่ปรับเพิ่ม คะแนน valid หลังปรับเป็นคะแนนเลือกโมเดล ไม่ใช่ผลทดสอบอิสระ


In [9]:
# Tune a single post-training decision bias from validation logits.
valid_result = trainer.predict(valid_ds)
valid_logits = valid_result.predictions[0] if isinstance(valid_result.predictions, tuple) else valid_result.predictions
valid_y = valid_result.label_ids
POS_ID = label2id['pos']
deltas = np.arange(0.0, 1.01, 0.1)
bias_rows = []
for delta in deltas:
    adjusted = valid_logits.copy()
    adjusted[:, POS_ID] -= delta
    pred = adjusted.argmax(axis=-1)
    report = classification_report(valid_y, pred, labels=list(range(len(LABELS))), target_names=LABELS, output_dict=True, zero_division=0)
    bias_rows.append({
        'pos_logit_subtract': float(delta),
        'macro_f1': f1_score(valid_y, pred, labels=list(range(len(LABELS))), average='macro', zero_division=0),
        'pos_precision': report['pos']['precision'], 'pos_recall': report['pos']['recall'],
        'pos_f1': report['pos']['f1-score'], 'neu_f1': report['neu']['f1-score']
    })
bias_trials = pd.DataFrame(bias_rows)
display(bias_trials)
# Maximize macro-F1; if tied, choose the smallest adjustment.
best_delta = float(bias_trials.sort_values(['macro_f1', 'pos_logit_subtract'], ascending=[False, True]).iloc[0]['pos_logit_subtract'])
print('Selected pos-logit subtraction:', best_delta)
bias_trials.to_csv(OUTPUT_DIR / f'{RUN_ID}_bias_trials_valid.csv', index=False)

all_errors = []
for name, dataset in [('valid', valid_ds), ('sample_labeled', sample_ds)]:
    result = valid_result if name == 'valid' else trainer.predict(dataset)
    logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    adjusted = logits.copy()
    adjusted[:, POS_ID] -= best_delta
    pred = adjusted.argmax(axis=-1)
    y = result.label_ids
    print('\n', name, '| pos bias:', best_delta)
    print(classification_report(y, pred, labels=list(range(len(LABELS))), target_names=LABELS, digits=4, zero_division=0))
    print(pd.DataFrame(confusion_matrix(y, pred, labels=list(range(len(LABELS)))), index=LABELS, columns=LABELS))
    records = frames[name].copy()
    records['pred'] = [id2label[int(i)] for i in pred]
    records['split'] = name
    records['run_id'] = RUN_ID
    records['pos_logit_subtract'] = best_delta
    records.to_csv(OUTPUT_DIR / f'{name}_predictions.csv', index=False)
    all_errors.append(records.loc[records['category'] != records['pred']])
pd.concat(all_errors, ignore_index=True).to_csv(OUTPUT_DIR / 'errors.csv', index=False)
trainer.save_model(str(OUTPUT_DIR / 'best_model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'best_model'))
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps({
    'model': MODEL_NAME, 'seed': SEED, 'max_length': MAX_LENGTH,
    'class_weighting': 'balanced inverse-frequency weights computed from train only',
    'class_weights': {label: float(class_weights[i]) for i, label in enumerate(LABELS)},
    'pos_logit_subtract': best_delta,
    'train_epochs': 3, 'learning_rate': 2e-5, 'batch_size': 8,
    'gradient_accumulation_steps': 2, 'preprocessing': 'HTML unescape, whitespace, newmm, space marker',
    'best_checkpoint': trainer.state.best_model_checkpoint, 'best_valid_macro_f1': trainer.state.best_metric,
    'torch': torch.__version__, 'transformers': transformers.__version__,
}, ensure_ascii=False, indent=2), encoding='utf-8')


,pos_logit_subtract,macro_f1,pos_precision,pos_recall,pos_f1,neu_f1
0,0.0,0.660799,0.551913,0.636555,0.591220,0.762186
1,0.1,0.658943,0.552830,0.615546,0.582505,0.763476
2,0.2,0.657624,0.555773,0.596639,0.575481,0.764127
3,0.3,0.659375,0.573805,0.579832,0.576803,0.770675
4,0.4,0.658375,0.586667,0.554622,0.570194,0.773848
5,0.5,0.659018,0.610169,0.529412,0.566929,0.780247
6,0.6,0.658340,0.620865,0.512605,0.561565,0.783462
7,0.7,0.657850,0.637602,0.491597,0.555160,0.787079
8,0.8,0.653978,0.641618,0.466387,0.540146,0.787168
9,0.9,0.655287,0.672897,0.453782,0.542033,0.792194


Selected pos-logit subtraction: 0.0

 valid | pos bias: 0.0
              precision    recall  f1-score   support

         pos     0.5519    0.6366    0.5912       476
         neu     0.8243    0.7088    0.7622      1456
         neg     0.7741    0.8490    0.8098       682
           q     0.3500    0.7636    0.4800        55

    accuracy                         0.7329      2669
   macro avg     0.6251    0.7395    0.6608      2669
weighted avg     0.7531    0.7329    0.7380      2669

     pos   neu  neg   q
pos  303   130   36   7
neu  233  1032  130  61
neg   13    80  579  10
q      0    10    3  42



 sample_labeled | pos bias: 0.0
              precision    recall  f1-score   support

         pos     0.8333    0.9524    0.8889        21
         neu     0.9762    0.8542    0.9111        48
         neg     0.8214    0.8846    0.8519        26
           q     0.8333    1.0000    0.9091         5

    accuracy                         0.8900       100
   macro avg     0.8661    0.9228    0.8902       100
weighted avg     0.8988    0.8900    0.8909       100

     pos  neu  neg  q
pos   20    0    1  0
neu    3   41    4  0
neg    1    1   23  1
q      0    0    0  5


776

### สร้างไฟล์ส่งจากโมเดลที่ fine-tune

รันเมื่อพร้อมสร้าง prediction; เปลี่ยน GROUP_ID ก่อนส่งจริง ผลนี้ใช้ checkpoint ที่เลือกด้วย valid

In [ ]:
assert GROUP_ID != 'G00', 'กรุณาเปลี่ยน GROUP_ID เป็นหมายเลขกลุ่มก่อนสร้างไฟล์ส่ง'
result = trainer.predict(test_ds)
logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
logits[:, POS_ID] -= best_delta
pred = np.argmax(logits, axis=-1)
submission = pd.DataFrame({'id': frames['test']['id'], 'category': [id2label[int(i)] for i in pred]})
assert submission['id'].equals(frames['test']['id']) and submission['category'].isin(LABELS).all()
PREDICTIONS_DIR = PROJECT_ROOT / 'predictions'
PREDICTIONS_DIR.mkdir(exist_ok=True)
output = PREDICTIONS_DIR / f'{GROUP_ID}_{RUN_ID}_wangchanberta.csv'
submission.to_csv(output, index=False, encoding='utf-8')
print('Saved:', output)


Saved: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/predictions/G00_run7_pos_neu_logit_bias_wangchanberta.csv
